# nexus-decision-4b: evaluation on Google Colab

This notebook merges the trained LoRA adapter into Qwen3.5-4B and runs the evaluation on a GPU. It
writes raw letter logits for the base and the tuned model. Calibration and the report run locally
(`eval/report.py`), so they always use the same code as the release.

**Before you start**

1. On your machine, run `bash scripts/pack_colab_eval.sh` and upload
   `artifacts/colab/nexus-eval.zip` to `MyDrive/nexus-decision-4b/`.
2. The adapter must be at `MyDrive/nexus-decision-4b/adapters/run1/final`, where the training
   notebook saved it.
3. Choose a GPU runtime (L4 or A100).

Results are written straight to `MyDrive/nexus-decision-4b/eval/`. Every evaluation cell can be
rerun after a disconnect, and it skips the items that are already done.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

DRIVE = '/content/drive/MyDrive/nexus-decision-4b'
ADAPTER = f'{DRIVE}/adapters/run1/final'
EVAL_DIR = f'{DRIVE}/eval'
WORK = '/content/nexus'
FUSED = '/content/fused/nexus-decision-4b'

!rm -rf {WORK} && mkdir -p {WORK} {EVAL_DIR}
!unzip -q -o {DRIVE}/nexus-eval.zip -d {WORK}
# Results go straight to Drive, so they survive a disconnect.
!rm -rf {WORK}/artifacts/eval && ln -s {EVAL_DIR} {WORK}/artifacts/eval
!ls {WORK} && ls -la {ADAPTER}

## Install

`flash-linear-attention` provides the fast Gated DeltaNet kernels.

In [ ]:
!pip install -q "transformers==5.22.2" "accelerate>=1.0" pydantic flash-linear-attention
# Colab ships a torchao version that newer Hugging Face libraries reject; it is not needed here.
!pip uninstall -y -q torchao

## 1. Merge the adapter (about 3 minutes)

Downloads Qwen/Qwen3.5-4B (about 9 GB) and adds the LoRA deltas to its weights. This is the same
script that produces the release checkpoint locally.

In [ ]:
!cd {WORK} && python train/fuse_export.py --adapters {ADAPTER} --out {FUSED}

## 2. Fast mode: tuned and base model

Each line prints progress every 100 items. Calib and test are 804 and 1,786 questions, each
scored in the canonical and the reversed option order.

In [ ]:
EV = f'cd {WORK} && python -m eval.run_eval --backend hf --mode fast'
!{EV} --name tuned --model {FUSED} --split calib 2>&1 | grep -v Warning
!{EV} --name tuned --model {FUSED} --split test 2>&1 | grep -v Warning
!{EV} --name base --model Qwen/Qwen3.5-4B --split calib 2>&1 | grep -v Warning
!{EV} --name base --model Qwen/Qwen3.5-4B --split test 2>&1 | grep -v Warning

In [ ]:
# Quick look (uncalibrated accuracy). The full report runs locally.
import glob
import json
import os
import pathlib

import numpy as np

for p in sorted(glob.glob(f'{EVAL_DIR}/*_fast.jsonl')):
    rs = [json.loads(l) for l in pathlib.Path(p).read_text().splitlines()]
    acc = np.mean([r['labels'][int(np.argmax(r['logits']))] == r['ref'] for r in rs])
    print(f'{os.path.basename(p):32s} n={len(rs):5d}  acc={acc:.3f}')

## 3. Think mode (optional)

Think mode generates a reasoning trace before the answer, so it is much slower than fast mode.
Run the calib subset first and check the `s/item` figure before you start the test subset.

In [ ]:
EVT = f'cd {WORK} && python -m eval.run_eval --backend hf --mode think --model {FUSED} --name tuned'
!{EVT} --split calib --limit 150 2>&1 | grep -v Warning

In [ ]:
!{EVT} --split test --limit 300 2>&1 | grep -v Warning

## 4. Download

Download `MyDrive/nexus-decision-4b/eval/` (every `*.jsonl` file) into the project's
`artifacts/eval/` folder, then on your machine run:

```bash
uv run python -m eval.report --name base
uv run python -m eval.report --name tuned
```